# M - Model: Pemodelan Clustering Desa IDM 2024
## Kelompok 3 - II4013 Data Analytics

Notebook ini berfokus pada pengelompokan (*clustering*) desa secara tidak terbimbing (*unsupervised*) berdasarkan 3 dimensi utama IDM: Indeks Ketahanan Sosial (IKS), Indeks Ketahanan Ekonomi (IKE), dan Indeks Ketahanan Lingkungan (IKL).

### Alur Pengerjaan:
1. **Load Data**: Membaca dataset bersih dari `data/processed/idm_2024_modeling.csv`.
2. **Feature Scaling**: Menstandardisasi pilar indeks komposit menggunakan `StandardScaler`.
3. **Elbow Method & Silhouette Score**: Menganalisis jumlah klaster optimal (K).
4. **K-Means Clustering**: Melatih model K-Means dengan K terbaik.
5. **Visualisasi Klaster**: Membangun visualisasi klaster dalam bentuk 2D (IKE × IKL) dan 3D.
6. **Profiling Klaster**: Menganalisis dan menginterpretasikan karakteristik masing-masing klaster desa.
7. **Export Model & Data**: Menyimpan model K-Means dan Scaler ke berkas `.pkl` untuk Streamlit, serta menyimpan dataset terklaster.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from mpl_toolkits.mplot3d import Axes3D
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# Konfigurasi grafik
pd.set_option('display.max_columns', 50)
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.figsize': (10, 5),
    'axes.titlesize': 13,
    'axes.labelsize': 11
})
sns.set_theme(style='whitegrid', palette='muted')

print('[OK] Library siap.')

In [ ]:
BASE_DIR = os.path.dirname(os.getcwd())
PROCESSED_DATA_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'idm_2024_modeling.csv')
CLUSTERED_DATA_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'idm_2024_clustered.csv')
MODELS_DIR = os.path.join(BASE_DIR, 'models')

df = pd.read_csv(PROCESSED_DATA_PATH, dtype={'KODE_DESA': str})
feature_cols = ['IKS_2024', 'IKE_2024', 'IKL_2024']
df_cluster = df[feature_cols].copy()
print(f'Memuat data cluster: {df_cluster.shape[0]:,} baris x {df_cluster.shape[1]} kolom')
df_cluster.head(3)

In [ ]:
# Standardisasi fitur agar kontribusi pilar indeks seimbang
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster)

print('Statistik deskriptif setelah standardisasi:')
display(pd.DataFrame(X_scaled, columns=feature_cols).describe().round(4))

In [ ]:
# Menggunakan sampel untuk efisiensi komputasi skor silhouette
np.random.seed(42)
sample_size = min(15000, len(X_scaled))
sample_idx = np.random.choice(len(X_scaled), size=sample_size, replace=False)
X_sample = X_scaled[sample_idx]

k_range = range(2, 9)
inertia_list = []
silhouette_list = []

print('Mencari jumlah klaster (K) optimal...')
for k in k_range:
    kmeans_model = KMeans(n_clusters=k, init='k-means++', n_init=20, max_iter=500, random_state=42)
    labels = kmeans_model.fit_predict(X_sample)
    inertia_list.append(kmeans_model.inertia_)
    silhouette_list.append(silhouette_score(X_sample, labels))
    print(f'K={k} selesai: WCSS={kmeans_model.inertia_:.2f}, Silhouette Score={silhouette_list[-1]:.4f}')

best_k = list(k_range)[int(np.argmax(silhouette_list))]
print(f'\n[BEST] Jumlah klaster optimal berdasarkan Silhouette Score: K={best_k}')

# Plotting Elbow Method & Silhouette Score
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(k_range), inertia_list, marker='o', color='#3B5998', linewidth=2)
axes[0].set_title('Metode Elbow (WCSS)')
axes[0].set_xlabel('Jumlah Klaster (K)')
axes[0].set_ylabel('Inertia (WCSS)')

axes[1].plot(list(k_range), silhouette_list, marker='o', color='#D1495B', linewidth=2)
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Jumlah Klaster (K)')
axes[1].set_ylabel('Silhouette Score')
plt.tight_layout()
plt.show()

In [ ]:
# Melatih model K-Means dengan K optimal
print(f'Melatih model K-Means dengan K={best_k}...')
kmeans = KMeans(n_clusters=best_k, init='k-means++', n_init=20, max_iter=500, random_state=42)
df['KLASTER'] = kmeans.fit_predict(X_scaled) + 1  # 1-indexed
df['KLASTER_LABEL'] = df['KLASTER'].apply(lambda x: f'Klaster {x}')

sil_final = silhouette_score(X_scaled, df['KLASTER'] - 1, sample_size=min(10000, len(X_scaled)), random_state=42)
print(f'Final Silhouette Score (sampel 10k): {sil_final:.4f}')
print('\nSebaran jumlah desa tiap klaster:')
print(df['KLASTER_LABEL'].value_counts().sort_index())

In [ ]:
# Visualisasi 2D Scatter Plot IKE vs IKL
palette = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A', '#F4A261', '#264653']
centroids = scaler.inverse_transform(kmeans.cluster_centers_)

plt.figure(figsize=(10, 6))
for cluster in sorted(df['KLASTER'].unique()):
    mask = df['KLASTER'] == cluster
    plt.scatter(df.loc[mask, 'IKL_2024'], df.loc[mask, 'IKE_2024'],
                s=10, alpha=0.3, color=palette[(cluster - 1) % len(palette)], label=f'Klaster {cluster}')
plt.scatter(centroids[:, 2], centroids[:, 1], marker='*', s=250, color='black', edgecolor='white', label='Centroid')
plt.title(f'Visualisasi Klaster Desa (K={best_k}): IKL vs IKE')
plt.xlabel('Indeks Ketahanan Lingkungan (IKL)')
plt.ylabel('Indeks Ketahanan Ekonomi (IKE)')
plt.legend()
plt.tight_layout()
plt.show()

# Visualisasi 3D Scatter Plot
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
for cluster in sorted(df['KLASTER'].unique()):
    mask = df['KLASTER'] == cluster
    ax.scatter(df.loc[mask, 'IKL_2024'], df.loc[mask, 'IKE_2024'], df.loc[mask, 'IKS_2024'],
               s=8, alpha=0.2, color=palette[(cluster - 1) % len(palette)], label=f'Klaster {cluster}')
ax.scatter(centroids[:, 2], centroids[:, 1], centroids[:, 0], marker='*', s=300, color='black', edgecolor='white', label='Centroid')
ax.set_title(f'Visualisasi 3D Klaster Desa IDM 2024 (K={best_k})')
ax.set_xlabel('IKL 2024')
ax.set_ylabel('IKE 2024')
ax.set_zlabel('IKS 2024')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Profil rata-rata tiap klaster
profil_klaster = df.groupby('KLASTER_LABEL')[feature_cols + ['NILAI_IDM_2024', 'jumlah_rekomendasi', 'total_nilai_rekomendasi']].mean().reset_index()
print('--- Profil Rata-rata Variabel per Klaster ---')
display(profil_klaster)

# Menghitung komposisi STATUS_IDM_2024 pada tiap klaster
komposisi_status = pd.crosstab(df['KLASTER_LABEL'], df['STATUS_IDM_2024'], normalize='index') * 100
print('\n--- Komposisi Status IDM per Klaster (%) ---')
display(komposisi_status)

# Plot stacked bar chart komposisi status
komposisi_status.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='viridis')
plt.title('Proporsi Status IDM pada Setiap Klaster')
plt.xlabel('Klaster')
plt.ylabel('Persentase (%)')
plt.legend(title='Status IDM', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Membuat folder models jika belum ada
os.makedirs(MODELS_DIR, exist_ok=True)

# Menyimpan Scaler dan Model KMeans
scaler_pkl_path = os.path.join(MODELS_DIR, 'scaler_clustering.pkl')
kmeans_pkl_path = os.path.join(MODELS_DIR, 'kmeans_model.pkl')

with open(scaler_pkl_path, 'wb') as f:
    pickle.dump(scaler, f)

with open(kmeans_pkl_path, 'wb') as f:
    pickle.dump(kmeans, f)

print(f'[SUCCESS] Scaler disimpan ke: {scaler_pkl_path}')
print(f'[SUCCESS] Model KMeans disimpan ke: {kmeans_pkl_path}')

# Menyimpan dataset yang sudah dilengkapi informasi klaster
df.to_csv(CLUSTERED_DATA_PATH, index=False, encoding='utf-8-sig')
print(f'[SUCCESS] Dataset terklaster disimpan ke: {CLUSTERED_DATA_PATH}')